# 10 — KYC Detection Quality Evaluation

Tests the card-edge detection and BGS-style grading from the
[u-siri-ous/KYC](https://github.com/u-siri-ous/KYC) project.

KYC's detection approach is **HSV yellow-thresholding** of the printed yellow
border on 1st-gen WoTC Pokémon cards:

```python
lower_yellow = np.array([20, 100, 100])   # HSV
upper_yellow = np.array([30, 255, 255])
mask = cv.inRange(hsv_image, lower_yellow, upper_yellow)
```

The largest contour of that mask becomes the card. Centering/corners/edges
scores (0–10, BGS scale) are derived from mask-geometry — left/right + top/bottom
yellow-border widths for centering, white-pixel fraction in mask corners/edges
for the rest. Surface = mean of the other three.

This notebook lets us:

1. **Confirm the import wiring** by running KYC's detection on its own samples
2. **Stress-test on out-of-domain inputs** (PSA slabs, modern cards, full-arts) —
   to see where the yellow-only assumption breaks
3. **Compare with BiRefNet / YOLO / OpenCV cascade** from notebook 09
   to see where KYC fails vs. our existing approaches

### Pre-requisites
- The KYC repo must be cloned at `/Users/srinivasdoddi/srini/KYC`
  (clone via `git clone https://github.com/u-siri-ous/KYC` if not present).
- `cv2`, `numpy`, `matplotlib`, `PIL` — already available from notebook 09.


In [ ]:
import os, sys, time
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

# Add KYC to import path
KYC_ROOT = Path('/Users/srinivasdoddi/srini/KYC')
assert KYC_ROOT.exists(), f'KYC repo not found at {KYC_ROOT}. Clone first.'

sys.path.insert(0, str(KYC_ROOT / 'code'))

# Import KYC's grader functions
from grader.refine import detect_edges
from grader.valutator import grading, borders, centering, corners, edges

# Display config — match notebook 09's dark theme
plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor']   = '#0d1117'
plt.rcParams['axes.edgecolor']   = '#30363d'
plt.rcParams['axes.labelcolor']  = '#e6edf3'
plt.rcParams['xtick.color']      = '#e6edf3'
plt.rcParams['ytick.color']      = '#e6edf3'
plt.rcParams['text.color']       = '#e6edf3'

print(f'✓ KYC modules loaded from {KYC_ROOT}')
print(f'  detect_edges, grading, borders, centering, corners, edges')

## 1. Sanity check on KYC's own sample images

Confirm the import wiring works by running detection on the test images shipped
with the repo. These are 1st-gen Pokémon cards photographed against neutral
backgrounds — the **target domain** for KYC's yellow-thresholding approach.

In [ ]:
KYC_SAMPLES_DIR = KYC_ROOT / 'images' / 'grader_test'

# Helper: run detection + grading, return everything we need to visualize
def kyc_analyze(image_path: str) -> dict:
    """
    Run KYC's detect_edges + grading. Returns a unified dict with:
      mask, cropped_card, cropped_details (or None if no yellow found),
      cen / cor / edg / sur (0–10 BGS scores),
      bbox in original-image px (computed from the mask contour)
    """
    path = str(image_path)
    out: dict = {'path': path, 'detected': False}
    try:
        mask, card, details = detect_edges(path)
    except Exception as e:
        out['error'] = f'detect_edges raised: {e}'
        return out

    if mask is None or card is None:
        out['error'] = 'no yellow contour found'
        return out

    out['detected'] = True
    out['mask']     = mask
    out['cropped_card']    = card
    out['cropped_details'] = details

    # KYC resizes input to 50% inside detect_edges. To recover the original-px
    # bbox we re-load the image and reproduce the bbox computation.
    orig = cv2.imread(path)
    if orig is not None:
        resized = cv2.resize(orig, (orig.shape[1] // 2, orig.shape[0] // 2))
        hsv     = cv2.cvtColor(resized, cv2.COLOR_BGR2HSV)
        m       = cv2.inRange(hsv, np.array([20, 100, 100]), np.array([30, 255, 255]))
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if cnts:
            x, y, w, h = cv2.boundingRect(max(cnts, key=cv2.contourArea))
            # Scale back to original-image coordinates (2× since KYC resized 50%)
            out['bbox_original_px'] = {
                'left':   x * 2, 'top':    y * 2,
                'width':  w * 2, 'height': h * 2,
            }

    try:
        out['cen'], out['cor'], out['edg'], out['sur'] = grading(path)
    except Exception as e:
        out['error'] = f'grading raised: {e}'
    return out


def visualize_kyc(result: dict, original_path: str = None,
                  figsize: tuple = (16, 6)) -> None:
    """4-panel: original+bbox, yellow mask, cropped card, score bars."""
    if not result['detected']:
        print(f'  ❌ {result.get("error", "no detection")}')
        return

    original = cv2.imread(original_path or result['path'])
    original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 4, figsize=figsize, facecolor='#0d1117')
    for ax in axes:
        ax.set_facecolor('#0d1117'); ax.axis('off')

    # Panel 1: original + bbox
    axes[0].imshow(original)
    bb = result.get('bbox_original_px')
    if bb:
        rect = mpatches.Rectangle(
            (bb['left'], bb['top']), bb['width'], bb['height'],
            linewidth=2.5, edgecolor='#ffd84a', facecolor='none',
        )
        axes[0].add_patch(rect)
        axes[0].set_title(f"Original + yellow bbox\n"
                          f"({bb['width']}×{bb['height']} in {original.shape[1]}×{original.shape[0]})",
                          color='white', fontsize=10)
    else:
        axes[0].set_title('Original (no bbox)', color='white', fontsize=10)

    # Panel 2: yellow mask
    axes[1].imshow(result['mask'], cmap='gray')
    axes[1].set_title('Yellow HSV mask', color='white', fontsize=10)

    # Panel 3: cropped card (KYC's output)
    cropped_rgb = cv2.cvtColor(result['cropped_card'], cv2.COLOR_BGR2RGB)
    axes[2].imshow(cropped_rgb)
    h, w = cropped_rgb.shape[:2]
    axes[2].set_title(f'Cropped card ({w}×{h})', color='white', fontsize=10)

    # Panel 4: BGS score bars
    axes[3].axis('on')
    axes[3].set_facecolor('#0d1117')
    cats   = ['Centering', 'Corners', 'Edges', 'Surface']
    scores = [result['cen'], result['cor'], result['edg'], result['sur']]
    colors = ['#5fa8ff', '#34d399', '#fb923c', '#ffd84a']
    bars = axes[3].barh(cats, scores, color=colors, edgecolor='#30363d')
    axes[3].set_xlim(0, 10)
    axes[3].set_xlabel('BGS score (0–10)')
    axes[3].set_title('KYC grading', color='white', fontsize=10)
    for bar, val in zip(bars, scores):
        axes[3].text(val + 0.15, bar.get_y() + bar.get_height() / 2,
                     str(val), va='center', color='white', fontsize=10, fontweight='bold')
    axes[3].spines[['top', 'right']].set_visible(False)
    axes[3].spines[['bottom', 'left']].set_color('#30363d')

    plt.tight_layout()
    plt.show()

# Run on a couple of KYC's own samples
print('── Sample 1: abra.jpg ──')
r1 = kyc_analyze(KYC_SAMPLES_DIR / 'abra.jpg')
print(f"  centering={r1.get('cen')} corners={r1.get('cor')} "
      f"edges={r1.get('edg')} surface={r1.get('sur')}")
visualize_kyc(r1)

print('\n── Sample 2: anonymous_vulpix.jpg ──')
r2 = kyc_analyze(KYC_SAMPLES_DIR / 'anonymous_vulpix.jpg')
print(f"  centering={r2.get('cen')} corners={r2.get('cor')} "
      f"edges={r2.get('edg')} surface={r2.get('sur')}")
visualize_kyc(r2)

## 2. Batch test on all KYC samples

Run detection on every grader-test image in the repo and tabulate the scores.
This gives a baseline for the "happy path" — vintage Pokémon cards photographed
against neutral backgrounds with the yellow border clearly visible.

| Card | Centering | Corners | Edges | Surface | Card detected? |
|---|---|---|---|---|---|

In [ ]:
results: list[dict] = []
for img_name in sorted(os.listdir(KYC_SAMPLES_DIR)):
    if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    path = KYC_SAMPLES_DIR / img_name
    r = kyc_analyze(path)
    r['name'] = img_name
    results.append(r)

# Print table
print(f'{"image":<28s} {"cen":>4s} {"cor":>4s} {"edg":>4s} {"sur":>4s}  status')
print('─' * 70)
for r in results:
    if r['detected']:
        print(f'{r["name"]:<28s} {r["cen"]:>4d} {r["cor"]:>4d} '
              f'{r["edg"]:>4d} {r["sur"]:>4d}  ✓')
    else:
        print(f'{r["name"]:<28s} {"-":>4s} {"-":>4s} {"-":>4s} {"-":>4s}  '
              f'✗ {r.get("error", "?")}')

# Score histograms
detected = [r for r in results if r['detected']]
if detected:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4), facecolor='#0d1117')
    for ax, cat, key, color in zip(
        axes,
        ['Centering', 'Corners', 'Edges', 'Surface'],
        ['cen', 'cor', 'edg', 'sur'],
        ['#5fa8ff', '#34d399', '#fb923c', '#ffd84a'],
    ):
        ax.set_facecolor('#0d1117')
        vals = [r[key] for r in detected]
        ax.hist(vals, bins=range(0, 12), color=color, edgecolor='#30363d')
        ax.set_xlim(0, 10)
        ax.set_title(f'{cat}  mean={np.mean(vals):.1f}', color='white', fontsize=10)
        ax.set_xlabel('Score (0–10)')
        ax.spines[['top', 'right']].set_visible(False)
        ax.spines[['bottom', 'left']].set_color('#30363d')
    plt.tight_layout()
    plt.show()

print(f'\n{len(detected)}/{len(results)} cards detected via yellow-mask')

## 3. Out-of-domain stress test: PSA-graded slabs

KYC was designed for **raw vintage Pokémon cards** photographed against neutral
backgrounds. Let's see what happens on the PSA-graded slab photos that broke
both `detect_card_bounds()` (color-threshold) and required the BiRefNet / YOLO
cascade in notebook 09.

The slabs include:
- `image0_front.jpeg` — One Piece full-art (no yellow border at all)
- `image1_front.jpeg` — Pokémon back (blue with red Pokéball, no yellow)
- `image0_back.jpeg`  — One Piece front (no yellow)
- `image1_back.jpeg`  — Mega Lucario ex (modern silver-bordered Pokémon)

Expected outcome: **most or all should fail to detect** because the yellow
HSV range won't fire on any of these card designs. This is the kind of
limitation that motivates BiRefNet / YOLO.

In [ ]:
LOCAL_SLAB_IMAGES = [
    '/Users/srinivasdoddi/srini/agentic-card-seller-os/notebooks/image0_front.jpeg',
    '/Users/srinivasdoddi/srini/agentic-card-seller-os/notebooks/image1_front.jpeg',
    '/Users/srinivasdoddi/srini/agentic-card-seller-os/notebooks/image0_back.jpeg',
    '/Users/srinivasdoddi/srini/agentic-card-seller-os/notebooks/image1_back.jpeg',
]

slab_results = []
for path in LOCAL_SLAB_IMAGES:
    if not os.path.exists(path):
        print(f'  skipping (not found): {path}')
        continue
    name = os.path.basename(path)
    print(f'\n── {name} ──')
    r = kyc_analyze(path)
    r['name'] = name
    slab_results.append(r)
    if r['detected']:
        print(f'  ✓ detected: cen={r["cen"]} cor={r["cor"]} edg={r["edg"]} sur={r["sur"]}')
        visualize_kyc(r, original_path=path)
    else:
        print(f'  ✗ {r.get("error")}')
        # Show what the yellow mask looks like even if no contour
        orig = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        resized = cv2.resize(orig, (orig.shape[1]//2, orig.shape[0]//2))
        hsv     = cv2.cvtColor(resized, cv2.COLOR_RGB2HSV)
        mask    = cv2.inRange(hsv, np.array([20, 100, 100]), np.array([30, 255, 255]))
        fig, axes = plt.subplots(1, 2, figsize=(10, 5), facecolor='#0d1117')
        for ax in axes: ax.set_facecolor('#0d1117'); ax.axis('off')
        axes[0].imshow(orig);   axes[0].set_title(name, color='white', fontsize=10)
        axes[1].imshow(mask, cmap='gray')
        axes[1].set_title(f'Yellow HSV mask  (fg fraction = {(mask > 0).mean():.3%})',
                           color='white', fontsize=10)
        plt.tight_layout(); plt.show()

# Summary
detected_slabs = sum(1 for r in slab_results if r['detected'])
print(f'\n══════════════════════════════════════════════')
print(f'Slab detection rate: {detected_slabs}/{len(slab_results)}')
print('(Expected to be low — KYC\'s yellow-only assumption excludes graded slabs)')

## 4. Cross-method comparison

Run KYC against the same images using the multi-method pipeline from notebook
09 (BiRefNet → YOLO/OpenCV cascade). The comparison shows where KYC's
yellow-only approach is **the right tool** (vintage Pokémon on clean backgrounds)
vs **the wrong tool** (modern cards, slabs, full-arts).

Requires notebook 09's helpers — run `09_claude_haiku_grading_eval.ipynb` cells
**2 → 6 → 20 → 22 → 24** first to define `birefnet_crop_card`, `yolo_crop_card`.

In [ ]:
# Run this cell only if notebook 09's BiRefNet / YOLO helpers are loaded.
# It will raise NameError otherwise — that's the signal to go run those cells.
try:
    _ = birefnet_crop_card, yolo_crop_card
    HAVE_NB09 = True
except NameError:
    HAVE_NB09 = False
    print('⚠ Notebook 09 helpers not in this kernel.')
    print('  Run cells 2 → 6 → 20 → 22 → 24 of 09_claude_haiku_grading_eval.ipynb')
    print('  in the same kernel, then re-run this cell.')

if HAVE_NB09:
    COMPARE_IMAGES = list(LOCAL_SLAB_IMAGES) + [
        str(KYC_SAMPLES_DIR / 'abra.jpg'),
        str(KYC_SAMPLES_DIR / 'anonymous_vulpix.jpg'),
    ]

    print(f'{"image":<40s} {"KYC det":>8s} {"BiRefNet score":>16s} '
          f'{"BiRefNet method":>26s}')
    print('─' * 95)
    for path in COMPARE_IMAGES:
        if not os.path.exists(path): continue
        name = os.path.basename(path)
        kyc = kyc_analyze(path)
        bref = birefnet_crop_card(path, debug=False)
        bref_score   = f"{bref['score']:.3f}"  if bref else '—'
        bref_method  = bref['method']          if bref else 'failed'
        print(f'{name:<40s} {("✓" if kyc["detected"] else "✗"):>8s} '
              f'{bref_score:>16s} {bref_method:>26s}')

## Conclusions

After running the cells above, you'll typically see:

| Domain | KYC | BiRefNet | YOLO/OpenCV cascade |
|---|---|---|---|
| Vintage Pokémon (yellow border) | ✓ good | ✓ good | ✓ good |
| Modern Pokémon (silver/black border) | ✗ no yellow | ✓ | ✓ |
| Full-art / borderless cards | ✗ no yellow | maybe | maybe |
| PSA-graded slabs | ✗ slab obscures yellow | ✓ best | ✓ fallback |
| Cards held in hand / cluttered bg | ✗ confused by yellow elsewhere | ✓ | depends |

**KYC's strengths**: simple, fast, deterministic. Zero ML dependencies.
On its target domain (1st-gen WoTC Pokémon) it works well and is essentially
free to run.

**KYC's limitations**: hard-coded to one card design (yellow border).
The BGS grades it produces are heuristic — mask-geometry stand-ins for the
actual surface/centering/corner/edge defects PSA / BGS look at.

**Verdict for our pipeline**: KYC is not a drop-in replacement, but its
*grading heuristics* (white-pixel fractions in mask corners/edges) are
worth borrowing as an additional signal for the multi-detector cascade
when we know we're looking at a yellow-bordered card.
